# Gold Layer (Data Science Table)
**Goal:** Prepare the Silver data strictly for the XGBoost model to predict Customer Card signups.

In [0]:
spark.conf.set("spark.sql.ansi.enabled", "false")
from pyspark.sql import functions as F
from pyspark.sql.window import Window
from pyspark.sql.functions import col, sum, countDistinct, datediff, current_date, when, count, row_number, isnull

#silver table filepaths
silver_table_customers = "dev.trailsport.silver_customers"
silver_table_products = "dev.trailsport.silver_products"
silver_table_sales = "dev.trailsport.silver_sales"
silver_table_sales_items = "dev.trailsport.silver_sales_items"

#gold table filepath
gold_table_science = "dev.trailsport.gold_science"

# 1. First pull Silver into a DataFrame

In [0]:
# Pull the differen tables in pd df
df_customers = spark.read.table(silver_table_customers)
df_products = spark.read.table(silver_table_products)
df_sales = spark.read.table(silver_table_sales)
df_sales_items = spark.read.table(silver_table_sales_items)



### 1.1 XGBoost requires numerical data, so 1 row per customer where we find counts and sums

In [0]:
#aggregate date data customer, delete obsolete
df_customers = df_customers.withColumn("Age",(datediff(current_date(),col("Birthdate"))/365).cast("integer"))\
    .withColumn("Days_is_Customer", datediff(current_date(),col("CustomerSince")))\
    .drop("BirthDate","CustomerSince")

#Force IDs to be strings before joining to prevent the BigInt crash
df_sales_items = df_sales_items.withColumn("SaleID", col("SaleID").cast("string")).withColumn("ProductID", col("ProductID").cast("string"))
df_sales = df_sales.withColumn("SaleID", col("SaleID").cast("string"))
df_products = df_products.withColumn("ProductID", col("ProductID").cast("string")).withColumn("Price", col("Price").cast("double"))

# Joining sales data with product data, Items -> Sales, Items -> Products. This a helper df.
df_sales_detailed = df_sales_items.join(df_sales, on="SaleID", how="left") \
                                  .join(df_products, on="ProductID", how="left")

#Calculate spending per customer. This will be in the gold table.
df_spending = df_sales_detailed.groupBy("CustomerID").agg(
    sum("Price").alias("Total_Amount_Spent"),
    countDistinct("SaleID").alias("Number_Of_Purchases"))

# Build recommendation, we find the #1 most purchased category for each customer
cat_counts = df_sales_detailed.groupBy("CustomerID", "ProductCategory").agg(count("*").alias("cat_count"))
window_spec = Window.partitionBy("CustomerID").orderBy(col("cat_count").desc())

favorite_cat_df = cat_counts.withColumn("rn", row_number().over(window_spec)) \
                            .filter(col("rn") == 1) \
                            .withColumnRenamed("ProductCategory", "Favorite_Product_Category") \
                            .select("CustomerID", "Favorite_Product_Category")

### 1.2 Merge all dataframes and clean

In [0]:
# Merge everything into our initial gold df
df_gold = df_customers.join(df_spending, on="CustomerID", how="left") \
                      .join(favorite_cat_df, on="CustomerID", how="left")


# Fill Nulls/empty cells (non-buyers)
df_gold = df_gold.fillna({
    "Total_Amount_Spent": 0.0,
    "Number_Of_Purchases": 0,
    "Favorite_Product_Category": "None"
})

#check if customer has card, if yes then 1. 
df_gold = df_gold.withColumn("Customer_HasLoyaltyCard", 
                             when(col("HasLoyaltyCard") == "Yes", 1).otherwise(0)) \
                 .drop("HasLoyaltyCard")


# Drop Leakage & IDs. For ML purposes, remove the email, otherwise AI cheats. Dataset gets stripped of cheating variables, identifiers, and useless noise.
leakage_and_id_cols = ["CustomerName", "ReceivedDiscountEmail"] 
df_gold = df_gold.drop(*leakage_and_id_cols)

# Dynamic Drop for >40% Missing Data
total_rows = df_gold.count()
# Calculate null counts for every column dynamically
null_counts = df_gold.select([sum(when(isnull(c), 1).otherwise(0)).alias(c) for c in df_gold.columns]).first().asDict()

cols_to_drop = [c for c, count in null_counts.items() if (count / total_rows) > 0.40]
print(f"Dropping columns with >40% missing data: {cols_to_drop}")
df_gold = df_gold.drop(*cols_to_drop)

# Categorical Encoding, to make sure its al 1s and 0s for each value (now column) and with that check yes or no
# We loop through text columns to create 1s and 0s manually so it saves as standard Delta columns
categorical_cols = ['Gender', 'MaritalStatus', 'OwnsHouse', 'Favorite_Product_Category', 'OwnsCar', 'Education', 'Occupation']

for c in categorical_cols:
    if c in df_gold.columns:
        # Get list of unique values in the column and puts it in memory
        distinct_vals = [row[0] for row in df_gold.select(c).distinct().dropna().collect()]
        
        # Create a binary column for each unique value and replaces any spaces with underscores so the database doesn't crash
        for val in distinct_vals:
            col_name = f"{c}_{str(val).replace(' ', '_')}"
            df_gold = df_gold.withColumn(col_name, when(col(c) == val, 1).otherwise(0)) #This creates the actual column. If the customer's original category was 'Running', put a 1 in this new column. If it was anything else, put a 0
        
        # Drop the original text column
        df_gold = df_gold.drop(c)

# Final drop of CustomerID (Models cannot learn from IDs!)
df_gold = df_gold.drop("CustomerID","ingest_time","Education", "Occupation", "OwnsCar")

Dropping columns with >40% missing data: ['PostalCode', 'CityCat']


### 1.3 Write the gold df into gold table

In [0]:
# Save as an optimized Delta table, overwriting previous day's run
df_gold.write \
    .format("delta") \
    .option("overwriteSchema", "true")\
    .mode("overwrite") \
    .saveAsTable(gold_table_science)

print(f"Success! ML Gold Table saved to: {gold_table_science}")

Success! ML Gold Table saved to: dev.trailsport.gold_science
